# Variational training

Every circuit seen so far fixes its angles in advance. A variational circuit leaves some of those angles as free parameters, and uses classical optimization to adjust them based on what is measured when running it, until a cost function is minimized or maximized. That combination of quantum circuit and classical optimizer is the basis of QAOA and of other hybrid algorithms. This notebook trains a small QAOA circuit to solve MaxCut, the problem of finding the maximum cut of a graph.

## Free parameters in a circuit

`polypus.Param(index)` marks an angle as a free parameter, instead of a fixed number: it is a reference to a position inside the parameter vector that gets bound at run time. `num_params` counts how many are still pending:

In [ ]:
import polypus

demo_qc = polypus.Circuit(1)
demo_qc.rx(0, polypus.Param(0))
print(demo_qc.num_params)

A circuit with pending parameters cannot be run directly:

In [ ]:
demo_qc.measure_all()
try:
    polypus.run_quantum_circuit(demo_qc, shots=100, infrastructure="local")
except ValueError as e:
    print(e)

The error itself points to the two ways to stop being pending: passing the values by hand with `to_qasm2(params)`, or training them. The rest of this notebook does the latter.

## The problem: MaxCut

MaxCut starts from a graph, nodes joined by edges, and looks for the way to split the nodes into two groups that maximizes the number of edges with one endpoint in each group. It is a classic combinatorial optimization problem. This notebook uses the smallest case with more than one edge: a path of three nodes, with edges between 0 and 1, and between 1 and 2.

In [ ]:
edges = [(0, 1), (1, 2)]


def cut_value(bitstring):
    bits = bitstring[::-1]  # bits[i] = value of qubit i, as in 04a
    return sum(1 for i, j in edges if bits[i] != bits[j])


best_cut = 0
for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            best_cut = max(best_cut, cut_value(f"{a}{b}{c}"))

print(best_cut)

With the three edges checked by hand, the best possible cut is 2, for example separating node 1 from nodes 0 and 2.

## Building the circuit: one-layer QAOA

QAOA starts from a uniform superposition over the eight possible partitions, with an H on each qubit. Then it alternates two layers: a cost layer, which applies a different phase depending on whether the two endpoints of each edge agree or differ, with the `rzz` gate seen in [`02a`](02a_gates_and_circuits.ipynb), and a mixing layer, which redistributes the amplitude between partitions with an `rx` on each qubit. Both layers each share a single angle, `gamma` for the cost and `beta` for the mixing, the two parameters that get trained:

In [ ]:
Param = polypus.Param

qc = polypus.Circuit(3)
qc.h(0).h(1).h(2)  # uniform superposition over the 8 partitions

qc.rzz(0, 1, Param(0))  # cost layer: one phase per edge
qc.rzz(1, 2, Param(0))

qc.rx(0, Param(1)).rx(1, Param(1)).rx(2, Param(1))  # mixing layer
qc.measure_all()

## Training with `train()`

`polypus.train()` takes the same infrastructure arguments as `run_quantum_circuit`, plus three of its own: `method`, the optimizer to use, `dimensions`, the number of free parameters, and `expectation_function`, a function that receives a single measured bitstring and returns its cost. Polypus takes care of running the circuit, measuring the shots, and averaging `expectation_function` over every bitstring obtained, weighted by how many times each one came up: no manual averaging needed. `cut_value`, already defined, fits as is.

The optimizer for this section is Differential Evolution, `DE`: it keeps a population of candidates per generation, and combines and mutates them looking to improve on the best one found. `DE` maximizes the cost function, so a bigger cut is directly a better value, with no need to flip the sign.

In [ ]:
de = polypus.DE(generations=60, population_size=20, tolerance=0.001, seed=7)

result = polypus.train(
    qc,
    de,
    shots=1000,
    n_qpus=1,
    dimensions=qc.num_params,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="maxcut-de",
)
print(result.best_params)
print(result.best_fitness)

## `TrainResult`'s fields

Besides `best_params` and `best_fitness`, the best point found and its cost, `result` carries other useful fields:

In [ ]:
print(result.iterations_run)
print(result.converged)
print(result.seed)
print(result.id)

`iterations_run` counts how many generations actually ran: with 47 out of the 60 possible, `DE` stopped early. `converged` confirms it: `True` means the accumulated improvement in `best_fitness` dropped below `tolerance` before using up the budget, instead of reaching the generation limit without stabilizing. `seed` is the effective seed used, the one passed explicitly in this case, useful for reproducing this exact run. `id` is the effective identifier: Polypus appends a unique suffix to the readable prefix passed in the call, to tell runs apart even if they share the same prefix. The result's `id` field, already seen in [`03a_local_execution.ipynb`](03a_local_execution.ipynb) for `run_quantum_circuit`, follows the same principle.

## Evaluating the result

`best_params` are just numbers: to run the circuit with them they need to be bound, just like in the error above, with `to_qasm2(params)` followed by `from_qasm2`:

In [ ]:
final_qc = polypus.Circuit.from_qasm2(qc.to_qasm2(result.best_params))
final_result = polypus.run_quantum_circuit(final_qc, shots=2000, infrastructure="local")
print(final_result.counts[0])

With those bitstrings, the average cut obtained can be computed and compared with the best possible cut:

In [ ]:
counts = final_result.counts[0]
total = sum(counts.values())
average_cut = sum(cut_value(bits) * n for bits, n in counts.items()) / total

print("measured average cut:", average_cut)
print("best possible cut:", best_cut)

The average cut lands below the optimum, around 82% of it: a single QAOA layer does not guarantee finding the maximum cut, it only favors partitions with more crossed edges over the rest. More layers, with more parameters to train, would bring the result closer to the optimum, at the cost of a bigger circuit.

With the same pattern seen in [`02b`](02b_qiskit_qasm_interop.ipynb), the trained circuit can be drawn:

In [ ]:
from qiskit import qasm2

qc_drawing = qasm2.loads(
    final_qc.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_drawing.draw("mpl")

## Summary

This notebook introduced variational circuits: angles marked as free parameters with `Param`, trained with `train()` to maximize a cost function, using Differential Evolution on a one-layer QAOA for MaxCut, and went through `TrainResult`'s fields.

The next section repeats this same problem with a different optimizer.

## Next step

Continue with: [`06b_pso_qng_optimizers.ipynb`](06b_pso_qng_optimizers.ipynb).